In [8]:
import os, h5py
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

from msfm.utils import catalog, lensing, files, observation
from deep_lss.utils import configuration

In [9]:
repo_dir = os.path.abspath("../../..")
conf_path = os.path.join(repo_dir, "configs/v16/default.yaml")
conf = files.load_config(conf_path)
dlss_conf = configuration.load_deep_lss_config("/users/athomsen/dlss/repos/y3-deep-lss/configs/v16/default/lensing/dlss.yaml")

n_side = conf["analysis"]["n_side"]
n_pix = hp.nside2npix(n_side)
# theta_fwhm = dlss_conf["scale_cuts"]["lensing"]["theta_fwhm"]
theta_fwhm = [60] * 4
survey_mask = files.get_mask(conf, nest_out=True)

26-05-13 14:47:04 configuratio INF   Loaded the config 


In [10]:
wl_gamma_map, _ = catalog.build_metacal_map_from_cat(conf)

des_fm_wl, obs_cls, footprint_pix = observation.forward_model_observation_map(
    wl_gamma_map=wl_gamma_map,
    conf=conf,
    apply_norm=False,
    with_padding=True,
    nest_in=False,
)

26-05-13 14:47:04   catalog.py INF   Loaded metacal maps from cache 


26-05-13 14:47:06 observation. INF   Forward modeling the weak lensing map 


In [11]:
fig_dir = os.path.join(repo_dir, "dev/notebooks/desy3/figures/wl")
os.makedirs(fig_dir, exist_ok=True)

# 10x10 deg gnomview patch in the DES footprint
_gnom_x_pix, _gnom_y_pix = 600, 600
_gnom_reso = 10 / _gnom_x_pix * 60  # arcmin/pixel
_gnom_rot = (90, -30, 0)

def plot_maps(tomo, obs_label="", cols=True, vmin=None, vmax=None, theta_fwhm=None):
    n_z = tomo.shape[1]
    grid = (1, n_z) if cols else (n_z, 1)
    subdir = "raw" if theta_fwhm is None else "smooth"

    maps_proc = []
    for i in range(n_z):
        m = np.zeros(n_pix)
        m[footprint_pix] = tomo[:, i]
        if theta_fwhm is not None:
            m[survey_mask == 0] = 0.0
            m = hp.smoothing(m, fwhm=np.radians(theta_fwhm[i] / 60), nest=True)
        m[survey_mask == 0] = hp.UNSEEN
        maps_proc.append(m)

    # Mollview
    plt.figure(figsize=(6 * n_z, 4) if cols else (6, 4 * n_z))
    for i, m in enumerate(maps_proc):
        clim = {} if vmin is None else {"min": vmin[i], "max": vmax[i]}
        hp.mollview(m, title=f"metacal bin {i+1}", sub=(*grid, i + 1), notext=True, xsize=2000, nest=True, cmap="plasma", **clim)
    plt.suptitle(obs_label, y=1.02)
    plt.tight_layout()
    out_dir = os.path.join(fig_dir, subdir, "mollview")
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, f"{obs_label}_wl_kappa_maps.png"), dpi=150, bbox_inches="tight")
    plt.close()

    # Gnomview
    fig, axes = plt.subplots(*grid, figsize=(4 * n_z, 4) if cols else (4, 4 * n_z), squeeze=False)
    axes = axes.flatten()
    for i, m in enumerate(maps_proc):
        patch = hp.gnomview(m, nest=True, rot=_gnom_rot, reso=_gnom_reso,
                            xsize=_gnom_x_pix, ysize=_gnom_y_pix,
                            return_projected_map=True, no_plot=True)
        plt.close()  # close any figure healpy opened internally
        clim = {} if vmin is None else {"vmin": vmin[i], "vmax": vmax[i]}
        axes[i].imshow(patch, origin="lower", cmap="plasma", **clim)
        axes[i].set_title(f"metacal bin {i+1}")
        axes[i].axis("off")
    fig.suptitle(obs_label, y=1.02)
    fig.tight_layout()
    out_dir = os.path.join(fig_dir, subdir, "gnomview")
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, f"{obs_label}_wl_kappa_maps.png"), dpi=150, bbox_inches="tight")
    plt.close()

In [12]:
cosmo_grid_obs_file = "/users/athomsen/scratch/deep_lss/data/v16/no_sc/obs/fiducial_bench_obs_maps.h5"

with h5py.File(cosmo_grid_obs_file, "r") as f:
    cg_fm_wl = f["obs/maps"][:, :, :4]

vmin = np.quantile(cg_fm_wl, 0.01, axis=(0, 1))
vmax = np.quantile(cg_fm_wl, 0.99, axis=(0, 1))

for i in range(10):
    plot_maps(cg_fm_wl[i], obs_label=f"CosmoGrid_{i}", vmin=vmin, vmax=vmax)

plot_maps(des_fm_wl, obs_label="DESy3", vmin=vmin, vmax=vmax)

/tmp/ipykernel_143475/2849888168.py:30: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [13]:
n_real, n_fp, n_z_wl = cg_fm_wl.shape
smoothed_fp = np.zeros((n_fp, n_z_wl))
for j in range(n_z_wl):
    m = np.zeros(n_pix)
    m[footprint_pix] = cg_fm_wl[0, :, j]
    m[survey_mask == 0] = 0.0
    m = hp.smoothing(m, fwhm=np.radians(theta_fwhm[j] / 60), nest=True)
    smoothed_fp[:, j] = m[footprint_pix]

vmin_smooth = np.quantile(smoothed_fp, 0.01, axis=0)
vmax_smooth = np.quantile(smoothed_fp, 0.99, axis=0)
del smoothed_fp

for i in range(10):
    plot_maps(cg_fm_wl[i], obs_label=f"CosmoGrid_{i}", vmin=vmin_smooth, vmax=vmax_smooth, theta_fwhm=theta_fwhm)

plot_maps(des_fm_wl, obs_label="DESy3", vmin=vmin_smooth, vmax=vmax_smooth, theta_fwhm=theta_fwhm)

/tmp/ipykernel_143475/2849888168.py:30: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
